# NCES Public School Data Download

### Importing necessary libraries

In [15]:
import time, os
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

### Setting up state mappings, columns to note, and chrome options for Selenium driver

In [16]:
# FIPS code for each State and Territory to use in URL construction for downloading school data for each state
state_FIPS = {
    'Alabama': '01',
    'Alaska': '02',
    'Arizona': '04',
    'Arkansas': '05',
    'California': '06',
    'Colorado': '08',
    'Connecticut': '09',
    'Delaware': '10',
    'DC': '11',
    'Florida': '12',
    'Georgia': '13',
    'Hawaii': '15',
    'Idaho': '16',
    'Illinois': '17',
    'Indiana': '18',
    'Iowa': '19',
    'Kansas': '20',
    'Kentucky': '21',
    'Louisiana': '22',
    'Maine': '23',
    'Maryland': '24',
    'Massachusetts': '25',
    'Michigan': '26',
    'Minnesota': '27',
    'Mississippi': '28',
    'Missouri': '29',
    'Montana': '30',
    'Nebraska': '31',
    'Nevada': '32',
    'New Hampshire': '33',
    'New Jersey': '34',
    'New Mexico': '35',
    'New York': '36',
    'North Carolina': '37',
    'North Dakota': '38',
    'Ohio': '39',
    'Oklahoma': '40',
    'Oregon': '41',
    'Pennsylvania': '42',
    'Rhode Island': '44',
    'South Carolina': '45',
    'South Dakota': '46',
    'Tennessee': '47',
    'Texas': '48',
    'Utah': '49',
    'Vermont': '50',
    'Virginia': '51',
    'Washington': '53',
    'West Virginia': '54',
    'Wisconsin': '55',
    'Wyoming': '56',
    'American Samoa': '60',
    'Guam': '66',
    'Northern Mariana Islands': '69',
    'Puerto Rico': '72',
    'U.S. Virgin Islands': '78'
}

keep_cols = ['NCES School ID', 'Low Grade', 'High Grade', 'School Name', 'District', 'County Name', 'Street Address', 'City',
             'State', 'ZIP', 'Phone', 'Locale Code', 'Charter', 'Students', 'Teachers', 'Free Lunch', 'Reduced Lunch', 'Type', 'Status']

final_cols = ['NCES School ID', 'Low Grade', 'High Grade', 'Account Name', 'School District', 'County Name', 'Billing Street',
              'Billing City', 'Billing State', 'Billing ZIP', 'Phone', 'School Environment', 'School Type',
              'Number of Students Served', 'Number of Teachers', 'Free Lunch', 'Reduced Lunch', 'Type', 'NCES Status']

# Set download directory
download_dir = f'{os.getcwd()}\\pub_downloads'
os.makedirs(download_dir, exist_ok=True) # Ensure the download directory exists

# Configure Chrome options for automatic download
chrome_options = webdriver.ChromeOptions()
prefs = {"download.default_directory": download_dir} # Establish and add download directory preference
chrome_options.add_experimental_option("prefs", prefs)
chrome_options.add_argument("--headless=new")  # Run Chrome in headless mode (optional)

### Function to Standardize Columns Kept

In [17]:
def create_sd_csv(excel_file, state):
    df = pd.concat(excel_file, ignore_index=True)

    # Change column names to values in keep_cols
    df.columns = df.loc[5]

    # NCES flags a column with a trailing asterisk when that column's data comes from a different (usually earlier) year than the rest of the row.
    # Useful for pipeline logging later to note per-state/per-field data-year mismatches; not acted on yet.
    # flagged_columns = [col for col in df.columns if '*' in str(col)]
    # if flagged_columns:
    #     print(f'{state}: columns flagged as using data from a previous year: {flagged_columns}')

    df.columns = df.columns.astype(str).str.replace('*', '', regex=False) # Remove NCES's data-year asterisk flags so columns match keep_cols exactly
    df = df[keep_cols]
    df = df.loc[6:].reset_index(drop=True) # Get only rows containing school data, reset index

    # Create masks to handle empty values for different columns
    free_lunch_empty_mask = (df['Free Lunch'] == '–') | (df['Free Lunch'] == '†') | (df['Free Lunch'] == '‡')
    reduced_lunch_empty_mask = (df['Reduced Lunch'] == '–') | (df['Reduced Lunch'] == '†') | (df['Reduced Lunch'] == '‡')
    num_students_empty_mask = (df['Students'] == '–') | (df['Students'] == '†') | (df['Students'] == '‡')
    num_teachers_empty_mask = (df['Teachers'] == '–') | (df['Teachers'] == '†') | (df['Teachers'] == '‡')
    county_name_empty_mask = (df['County Name'] == '–') | (df['County Name'] == '†') | (df['County Name'] == '‡')
    ungraded_grade_empty_mask = (df['Low Grade'] == '–') & (df['High Grade'] == '–')

    # Assign empty string to empty values, convert non-empty values to appropriate data types
    df.loc[free_lunch_empty_mask, 'Free Lunch'] = ''
    df.loc[~free_lunch_empty_mask, 'Free Lunch'] = df.loc[~free_lunch_empty_mask, 'Free Lunch'].astype(float).astype(str).str.replace(r'\.0$', '', regex=True)

    df.loc[reduced_lunch_empty_mask, 'Reduced Lunch'] = ''
    df.loc[~reduced_lunch_empty_mask, 'Reduced Lunch'] = df.loc[~reduced_lunch_empty_mask, 'Reduced Lunch'].astype(float).astype(str).str.replace(r'\.0$', '', regex=True)

    df.loc[num_students_empty_mask, 'Students'] = ''
    df.loc[~num_students_empty_mask, 'Students'] = df.loc[~num_students_empty_mask, 'Students'].astype(float).astype(str).str.replace(r'\.0$', '', regex=True)

    df.loc[num_teachers_empty_mask, 'Teachers'] = ''
    df.loc[~num_teachers_empty_mask, 'Teachers'] = df.loc[~num_teachers_empty_mask, 'Teachers'].astype(float).astype(str).str.replace(r'\.0$', '', regex=True)

    df.loc[county_name_empty_mask, 'County Name'] = ''

    # Change all values in column Type to 'School'
    df['Type'] = 'School'

    # Create Locale mappings/masks and update Locale column based on Locale Code values
    locale_code_numeric = pd.to_numeric(df['Locale Code'], errors='coerce')

    urban = [11, 12, 13]
    urban_mask = locale_code_numeric.isin(urban)

    suburban = [21, 22, 23, 31, 32, 33]
    suburban_mask = locale_code_numeric.isin(suburban)

    rural = [41, 42, 43]
    rural_mask = locale_code_numeric.isin(rural)

    df.loc[urban_mask, 'Locale Code'] = 'Urban'
    df.loc[suburban_mask, 'Locale Code'] = 'Suburban'
    df.loc[rural_mask, 'Locale Code'] = 'Rural'

    # Blank out any code that isn't missing but also doesn't fall into one of the three categories
    unrecognized_locale_mask = ~(urban_mask | suburban_mask | rural_mask)
    df.loc[unrecognized_locale_mask, 'Locale Code'] = ''

    # Remove leading zeros from Low Grade and High Grade
    values = ['01','02','03','04','05','06','07','08','09']
    df.loc[df['Low Grade'].isin(values), 'Low Grade'] = df['Low Grade'].str.lstrip('0')
    df.loc[df['High Grade'].isin(values), 'High Grade'] = df['High Grade'].str.lstrip('0')

    df.loc[ungraded_grade_empty_mask, ['Low Grade', 'High Grade']] = '' # Assign empty string to ungraded schools

    numeric_high_grade = pd.to_numeric(df['High Grade'], errors='coerce')
    df.loc[numeric_high_grade > 12, 'High Grade'] = '12' # Change any High Grade values above 12 to 12

    # Change 'Status' values of 'Added', 'Changed Agency', and 'Reopened' to 'Open'
    change_status_to_open_mask = df['Status'].isin(['Added', 'Changed Agency', 'Reopened'])
    df.loc[change_status_to_open_mask, 'Status'] = 'Open'

    # Determine school types based on Charter status
    charter_mask = (df['Charter'] == 'Yes')

    df.loc[charter_mask, 'Charter'] = 'Public- Charter'
    df.loc[~charter_mask, 'Charter'] = 'Public'

    # Change state abbreviation to full name
    df['State'] = state

    # Update columns to final version we want
    df.columns = final_cols

    # Temp output
    df.to_csv(f'{download_dir}\\{state}_sd.csv', index=False)

### Main Loop to Retrieve, Standardize, and Save Files

In [ ]:
# Logic to navigate to School Data Excel file and download it
for state, fips in state_FIPS.items():
    # # Create Selenium driver instance
    driver = webdriver.Chrome(options=chrome_options)

    # Construct URL for each state using FIPS code
    url = f"""https://nces.ed.gov/ccd/schoolsearch/school_list.asp?Search=1&InstName=&SchoolID=&Address=&City=&State={fips}&Zip=&Miles=&County=&PhoneAreaCode=&Phone=&DistrictName=&DistrictID=&SchoolType=1&SchoolType=2&SchoolType=3&SchoolType=4&SpecificSchlTypes=all&IncGrade=-1&LoGrade=-1&HiGrade=-1"""

    print(f'Downloading public school data for {state}...')
    try:
        files_before_download = set(os.listdir(download_dir)) # Snapshot directory before download so we can identify exactly which file this state produces

        driver.get(url) # Navigate to the specified URL
        wait = WebDriverWait(driver, 10) # Initialize WebDriverWait with a timeout of 10 seconds

        # Check quickly whether this state has any results before committing to the full click/wait sequence
        try:
            excel_link = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CLASS_NAME, 'excelclass')))
        except TimeoutException:
            print(f'{state}: no public schools found, skipping')
            continue

        # print(f'Before clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows before clicking Excel link

        excel_link.click() # Click on link that opens window with Excel download link
        wait.until(EC.number_of_windows_to_be(2)) # Wait for the new window to actually open instead of a fixed sleep

        # print(f'After clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows after clicking Excel link to ensure new window opened

        # Switch to the new window that opened after clicking the Excel link
        driver.switch_to.window(driver.window_handles[-1])

        wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Download Excel File'))).click() # Click Excel download link in new window

        # Poll for the download to complete instead of a fixed sleep
        download_timeout = 30 # seconds to wait for the download to complete
        poll_interval = 0.5
        elapsed = 0
        new_files = []
        while elapsed < download_timeout:
            new_files = [f for f in os.listdir(download_dir) if f not in files_before_download and (f.endswith('.xlsx') or f.endswith('.xls'))]
            if new_files:
                break
            time.sleep(poll_interval)
            elapsed += poll_interval

        if new_files:
            downloaded_file = os.path.join(download_dir, new_files[0])
            try:
                excel_file = pd.read_html(downloaded_file) # Read the downloaded Excel file into a DataFrame

                print(f'{state}: {new_files[0]}')
                create_sd_csv(excel_file, state) # Create CSV file from the downloaded Excel file
            finally:
                os.remove(downloaded_file) # Always remove the original Excel file, even if parsing failed

    except Exception as e:
        print(f'Error downloading data for {state}: {e}')
    finally:
        print(f'{state} Download Complete\n')
        driver.quit()

### Testing Cell

In [18]:
test_states = ['Alabama', 'Alaska', 'Arizona', 'DC', 'Guam'] # Includes DC and a territory to exercise the zero-results skip path

for state in test_states:
    fips = state_FIPS[state]

    # Create Selenium driver instance
    driver = webdriver.Chrome(options=chrome_options)

    # Construct URL for each state using FIPS code
    url = f"""https://nces.ed.gov/ccd/schoolsearch/school_list.asp?Search=1&InstName=&SchoolID=&Address=&City=&State={fips}&Zip=&Miles=&County=&PhoneAreaCode=&Phone=&DistrictName=&DistrictID=&SchoolType=1&SchoolType=2&SchoolType=3&SchoolType=4&SpecificSchlTypes=all&IncGrade=-1&LoGrade=-1&HiGrade=-1"""

    print(f'Downloading public school data for {state}...')
    try:
        files_before_download = set(os.listdir(download_dir)) # Snapshot directory before download so we can identify exactly which file this state produces

        driver.get(url) # Navigate to the specified URL
        wait = WebDriverWait(driver, 10) # Initialize WebDriverWait with a timeout of 10 seconds

        # Check quickly whether this state has any results before committing to the full click/wait sequence
        try:
            excel_link = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CLASS_NAME, 'excelclass')))
        except TimeoutException:
            print(f'{state}: no public schools found, skipping')
            continue

        # print(f'Before clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows before clicking Excel link

        excel_link.click() # Click on link that opens window with Excel download link
        wait.until(EC.number_of_windows_to_be(2)) # Wait for the new window to actually open instead of a fixed sleep

        # print(f'After clicking Excel link: {len(driver.window_handles)}') # For Debugging: Check number of windows after clicking Excel link to ensure new window opened

        # Switch to the new window that opened after clicking the Excel link
        driver.switch_to.window(driver.window_handles[-1])

        wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Download Excel File'))).click() # Click Excel download link in new window

        # Poll for the download to complete instead of a fixed sleep
        download_timeout = 30 # seconds to wait for the download to complete
        poll_interval = 0.5
        elapsed = 0
        new_files = []
        while elapsed < download_timeout:
            new_files = [f for f in os.listdir(download_dir) if f not in files_before_download and (f.endswith('.xlsx') or f.endswith('.xls'))]
            if new_files:
                break
            time.sleep(poll_interval)
            elapsed += poll_interval

        if new_files:
            downloaded_file = os.path.join(download_dir, new_files[0])
            try:
                excel_file = pd.read_html(downloaded_file) # Read the downloaded Excel file into a DataFrame

                print(f'{state}: {new_files[0]}')
                create_sd_csv(excel_file, state) # Create CSV file from the downloaded Excel file
            finally:
                os.remove(downloaded_file) # Always remove the original Excel file, even if parsing failed

    except Exception as e:
        print(f'Error downloading data for {state}: {e}')
    finally:
        print(f'{state} Download Complete\n')
        driver.quit()

Alabama: ncesdata_CF014371.xls
Alabama Download Complete

Alaska: ncesdata_8383EBA5.xls
Alaska Download Complete

Arizona: ncesdata_16D9A764.xls
Arizona Download Complete

DC: ncesdata_49151B2B.xls
DC Download Complete

Guam: ncesdata_4893A37C.xls
Guam Download Complete

